# Ollama — Endpoint Tests

Tests all Ollama API endpoints on the DGX Spark host.

**Prerequisites:** Ollama running + a model loaded (`Actions → Ollama Deploy`) + SSH tunnel open.

```sh
ssh -L 11434:localhost:11434 -L 8888:localhost:8888 <user>@spark-79b7.local
```

In [ ]:
import requests, json

BASE = "http://localhost:11434"

def check(label, r):
    status = "\u2705" if r.ok else "\u274c"
    print(f"{status} {label}: HTTP {r.status_code}")
    if not r.ok:
        print(f"   {r.text[:300]}")
    return r

## What's loaded

In [ ]:
r = check("Loaded models (api/ps)", requests.get(f"{BASE}/api/ps"))
if r.ok:
    models = r.json().get("models", [])
    if models:
        for m in models:
            vram_gb = m.get("size_vram", 0) // 1024**3
            print(f"  - {m.get('name')} ({vram_gb} GB VRAM)")
    else:
        print("  No models currently loaded in VRAM")

## Available models

In [ ]:
r = check("Available models (v1/models)", requests.get(f"{BASE}/v1/models"))
if r.ok:
    models = r.json().get("data", [])
    print(f"  {len(models)} model(s) on disk")
    for m in models:
        print(f"  - {m.get('id')}")

## Chat completion

In [ ]:
# Set to whatever model is currently loaded (see 'What's loaded' above)
MODEL = "gpt-oss:20b"  # fastest on Spark (~58 tok/s)

r = check(f"Chat ({MODEL})", requests.post(f"{BASE}/v1/chat/completions", json={
    "model": MODEL,
    "messages": [{"role": "user", "content": "Respond with exactly one word: hello"}],
    "stream": False,
}))
if r.ok:
    content = r.json()["choices"][0]["message"]["content"]
    print(f"  Response: {content.strip()}")

## Tool calling

In [ ]:
MODEL = "llama3.3:70b-instruct-q4_K_M"  # best tool-calling model

r = check(f"Tool calling ({MODEL})", requests.post(f"{BASE}/v1/chat/completions", json={
    "model": MODEL,
    "messages": [{"role": "user", "content": "What is the weather in Paris?"}],
    "tools": [{
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        },
    }],
    "tool_choice": "auto",
    "stream": False,
}))
if r.ok:
    choice = r.json()["choices"][0]
    if choice.get("finish_reason") == "tool_calls":
        for tc in choice["message"].get("tool_calls", []):
            print(f"  Tool called: {tc['function']['name']}({tc['function']['arguments']})")
    else:
        print(f"  finish_reason: {choice.get('finish_reason')} — model may not support tool calling")

## Embeddings

In [ ]:
MODEL = "llama3.3:70b-instruct-q4_K_M"

r = check(f"Embeddings ({MODEL})", requests.post(f"{BASE}/v1/embeddings", json={
    "model": MODEL,
    "input": "The quick brown fox jumps over the lazy dog.",
}))
if r.ok:
    vec = r.json()["data"][0]["embedding"]
    print(f"  Embedding dim: {len(vec)} | first 4: {[round(v, 4) for v in vec[:4]]}")

## Summary

In [ ]:
results = [
    ("Loaded models",    requests.get(f"{BASE}/api/ps")),
    ("Available models", requests.get(f"{BASE}/v1/models")),
]
print("\n=== Summary ===")
for label, r in results:
    print(f"{'\u2705' if r.ok else '\u274c'} {label}: HTTP {r.status_code}")
failed = sum(1 for _, r in results if not r.ok)
print(f"\n{len(results) - failed}/{len(results)} connectivity checks passed")
print("(Inference cells require a model to be loaded — run Ollama Deploy first)")